# Install Dependencies

In [4]:
# install standard libraries
#%pip install tensorflow

In [5]:
#%pip install opencv-python

# Importing libraries

In [6]:
#import standard libraries
import cv2, random , os , numpy as np , matplotlib.pyplot as plt

In [7]:
# import tensorflow dependencies - functional api 
from tensorflow.keras.models import Model # Model(inp = [inp image, verifi img],out=[0,1])
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten 
# layer- custom layers, conv2d - convolution , dense - fully connected layer, max pooling - pull out layers and shrink 
#  input - define what were passing to model , flatten- flows data from layers

import tensorflow as tf


# CREATE FOLDERS

In [16]:
# create 3 folders(paths) : anchor(input) / positive(verification image) / negative(verification image)
#setup paths 

pos_path = os.path.join('data','positive') # data/positive
neg_path = os.path.join('data','negative') #data/negative
anchor_path = os.path.join('data','anchor') #data/anchor


In [23]:
# make directories 
'''
os.makedirs(pos_path)
os.makedirs(neg_path)
os.makedirs(anchor_path)
'''

'\nos.makedirs(pos_path)\nos.makedirs(neg_path)\nos.makedirs(anchor_path)\n'

## Collect positive and Anchors

In [18]:
# import uuuid(universally unique identifier) to generate unique image names 
import uuid 

In [19]:
# uuid1, uuid2, uuid3, uuid4, uuid5  - formats you can use  
uuid.uuid1()

UUID('f5d725fa-381e-11f1-9307-40490f13fa1a')

In [20]:
'{}.jpg'.format(uuid.uuid1())

'f7682dc9-381e-11f1-bc0d-40490f13fa1a.jpg'

In [21]:
os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) 

'data\\anchor\\f8920123-381e-11f1-bc0b-40490f13fa1a.jpg'

In [22]:
# connect to webcam 
import cv2
cap = cv2.VideoCapture(0)
while cap.isOpened(): # loop through every frame in webcam 
    ret , frame = cap.read() # returns val and actual frame

    frame = frame[120:120+250,200:200+250,:] # cut down frame to 250px - 250px 

    # collect anchors 
    if cv2.waitKey(1) & 0XFF == ord('a'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)


    # collect positives
    if cv2.waitKey(1) & 0XFF == ord('p'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(pos_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)



    # show image back to screen    
    cv2.imshow('Image Collection', frame) 
    
    # press q to break webcam connection
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break


#release the webcam 
cap.release()
# close the frame showing image 
cv2.destroyAllWindows()


In [24]:
# click a - anchor , p- positive 
#plt.imshow(frame)
frame.shape # 250px x 250px x 3 channels

(250, 250, 3)

In [28]:
# slicing to get right shape [yaxis,xaixs]
#plt.imshow(frame[120:120+250,200:200+250,:])

# LOAD AND PREPROCESS IMAGES 

In [29]:
# get image directories 
# 3 vars = grab all images in the specific folder 
anchor = tf.data.Dataset.list_files(anchor_path+'\*.jpg').take(90)# get every file with \xxxxxx.jpg only 90 images
positive = tf.data.Dataset.list_files(pos_path+'\*.jpg').take(90)# number of samples should be same it only loads path not actual images
negative = tf.data.Dataset.list_files(neg_path+'\*.jpg').take(90)

In [33]:
anchor_path+'\*.jpg'

'data\\anchor\\*.jpg'

In [35]:
dir_test = anchor.as_numpy_iterator()
dir_test.next() # get next file( image id )

b'data\\anchor\\41833586-381f-11f1-80e5-40490f13fa1a.jpg'

## PREPROCESSING - SCALE AND RESIZE 
fxn to load image and resize it  and convert image val to 0-255 to 0-1

In [45]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)  # read img using file path treeated using bytes like obj 
    img = tf.io.decode_jpeg(byte_img)          # decode jpeg -> load img 
    
    img = tf.image.resize(img,(100,100))    # resize img -> preprocessing (100px, 100px, 3 channels) as per paper 
    img = img / 255.0              # divide it by 255 so it performs scaling and it returnns the image 0-1
    return img

In [46]:
img = preprocess('data\\anchor\\41833586-381f-11f1-80e5-40490f13fa1a.jpg')

In [49]:
img.shape

TensorShape([100, 100, 3])

In [50]:
img.numpy().min()

np.float32(0.014705882)

In [51]:
img.numpy().max()

np.float32(0.89730394)

In [52]:
#plt.imshow(img)

## CREATE LABELLED DATASET 

In [53]:
# use 
tf.ones_like(1)

<tf.Tensor: shape=(), dtype=int32, numpy=1>

In [54]:
tf.ones_like([1,1,3,4,555.5,666.0]) # - > set of ones not different vals 

<tf.Tensor: shape=(6,), dtype=float32, numpy=array([1., 1., 1., 1., 1., 1.], dtype=float32)>

In [55]:
# pass anchot + pos img as input // otp -> array os ones 
# same for negatives -> otp 0 

In [57]:
tf.ones(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.], dtype=float32)>

In [65]:
tf.zeros(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.], dtype=float32)>

In [68]:
tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor))) # putting data into data loader 

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.float32, name=None)>

In [62]:
# anc +pos -> 1
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
# anc + neg -> 0
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [63]:
data

<_ConcatenateDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [70]:
samples  = data.as_numpy_iterator()

In [72]:
samples.next() # keep iterating to dpoints

(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg',
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',
 np.float32(1.0))

## BUILD TRAIN AND TEST PARTITION

In [73]:
'''(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg', # input 
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',  # valid 
 np.float32(1.0))'''
def preprocess_twin(input_image, validation_image, label):
    return (preprocess(input_image), preprocess(validation_image), label)

In [76]:
res = preprocess_twin(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg',
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',
 np.float32(1.0))

In [78]:
res[0] # converts 100 , 100 , 3

<tf.Tensor: shape=(100, 100, 3), dtype=float32, numpy=
array([[[0.76985294, 0.7776961 , 0.75808823],
        [0.7590686 , 0.76691175, 0.7473039 ],
        [0.76593137, 0.782598  , 0.75710785],
        ...,
        [0.8622549 , 0.89362746, 0.8504902 ],
        [0.8784314 , 0.9098039 , 0.8607843 ],
        [0.8627451 , 0.8862745 , 0.83137256]],

       [[0.7683824 , 0.7713235 , 0.73995095],
        [0.7588235 , 0.76862746, 0.74019605],
        [0.76862746, 0.77843136, 0.75      ],
        ...,
        [0.8757353 , 0.90661764, 0.85784316],
        [0.8620098 , 0.8870098 , 0.84117645],
        [0.8610294 , 0.88529414, 0.82892156]],

       [[0.76715684, 0.76715684, 0.72205883],
        [0.7767157 , 0.7769608 , 0.73382354],
        [0.7772059 , 0.77818626, 0.7409314 ],
        ...,
        [0.8661765 , 0.8904412 , 0.8512255 ],
        [0.8691176 , 0.8926471 , 0.85343134],
        [0.8762255 , 0.89093137, 0.84632355]],

       ...,

       [[0.73995095, 0.7352941 , 0.7169118 ],
        [0.73

In [79]:
res[2]

np.float32(1.0)

In [84]:
#plt.imshow(res[0])

In [85]:
# build data loader pipeline 
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size = 1024)

In [87]:
sample = data.as_numpy_iterator()

In [97]:
eg = sample.next()

In [98]:
#plt.imshow(eg[0])

In [99]:
#plt.imshow(eg[1])

In [100]:
eg[2]

np.float32(0.0)